# Notebook 02 — Preprocessing Pipeline

## Purpose
Build and demonstrate the full preprocessing pipeline on our dataset.
Every step is shown with a before/after example.

The reusable implementation lives in `src/preprocessing.py`.
This notebook imports from there — it does not redefine any functions.

## Steps Covered
1. Text cleaning
2. Section extraction
3. Sentence segmentation
4. Stopword removal
5. Keyword extraction (KeyBERT + YAKE)
6. Named Entity Recognition (spaCy NER)

In [3]:
# ══════════════════════════════════════════════════════════════════════════════
#  GLOBAL CONFIG — match this to whatever you set in Notebook 01
# ══════════════════════════════════════════════════════════════════════════════
DATASET_MODE = "full"   # "dev" | "full"
SEED = 42

import random, json, os, sys
random.seed(SEED)

# Make sure src/ is importable
sys.path.append(os.path.abspath(".."))

In [4]:
# Load the normalised splits saved by Notebook 01 — no re-downloading needed
data_dir = f"../data/processed_{DATASET_MODE}"

with open(f"{data_dir}/train.json", encoding="utf-8") as f:
    train_rows = json.load(f)

print(f"Loaded {len(train_rows)} train rows from {data_dir}/")
print(f"Fields in each row: {list(train_rows[0].keys())}")

Loaded 322961 train rows from ../data/processed_full/
Fields in each row: ['article', 'summary', 'domain', 'title', 'keywords']


In [6]:
from src.preprocessing import build_pipeline

# Load all NLP models once — reuse this object for all subsequent cells
pipeline = build_pipeline()

Loading spaCy model ...
Loading KeyBERT model ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading YAKE extractor ...
Pipeline ready.


## 1. Text Cleaning

Removes LaTeX commands, citation markers, URLs, figure references,
and non-ASCII characters. Normalises whitespace.

In [7]:
from src.preprocessing import clean_text

sample = random.choice(train_rows)
raw_text = sample["article"]

cleaned = clean_text(raw_text)

print("── BEFORE (first 600 chars) ─────────────────────────────────────────")
print(raw_text[:600])
print("\n── AFTER cleaning (first 600 chars) ────────────────────────────────")
print(cleaned[:600])
print(f"\nLength before: {len(raw_text):,} chars")
print(f"Length after : {len(cleaned):,} chars")
print(f"Reduction    : {100*(1 - len(cleaned)/len(raw_text)):.1f}%")

── BEFORE (first 600 chars) ─────────────────────────────────────────
in order to better understand the nature of star formation occurring in nearby starburst galaxies and to provide a foundation for studies of higher redshift systems , we have been analyzing the properties of the 20 starburst galaxies with the highest 5 ghz luminosities from condon , frayer , & broderick ( 1991 ) . 
 these galaxies form a sample of relatively nearby , northern hemisphere starbursts which is independent of the dust content and the morphology of the host galaxy . 
 an example of our data and analysis technique is given in smith et al . 
 ( 1995 ) , hereafter paper 1 . the complet

── AFTER cleaning (first 600 chars) ────────────────────────────────
in order to better understand the nature of star formation occurring in nearby starburst galaxies and to provide a foundation for studies of higher redshift systems , we have been analyzing the properties of the 20 starburst galaxies with the highest 5 ghz lu

## 2. Section Extraction

Uses regex heuristics to identify and extract named sections:
Abstract, Introduction, Methodology, Results, Discussion, Conclusion.

In the dev dataset these sections may not always be explicitly labelled
(since articles are abstract+conclusion only). On the full arXiv/PubMed
datasets, section headers are much more common.

In [8]:
from src.preprocessing import extract_sections

sections = extract_sections(cleaned)

print("── Extracted Sections ───────────────────────────────────────────────")
for name, content in sections.items():
    status = f"{len(content)} chars" if content.strip() else "NOT FOUND"
    preview = content[:200].strip() if content.strip() else ""
    print(f"\n[{name.upper()}]  ({status})")
    if preview:
        print(preview + ("..." if len(content) > 200 else ""))

── Extracted Sections ───────────────────────────────────────────────

[ABSTRACT]  (NOT FOUND)

[INTRODUCTION]  (81657 chars)
in order to better understand the nature of star formation occurring in nearby starburst galaxies and to provide a foundation for studies of higher redshift systems , we have been analyzing the proper...

[METHODOLOGY]  (NOT FOUND)

[RESULTS]  (NOT FOUND)

[DISCUSSION]  (NOT FOUND)

[CONCLUSION]  (NOT FOUND)


## 3. Sentence Segmentation

Uses spaCy's dependency parser to detect sentence boundaries.
More accurate than simple period-splitting — handles abbreviations,
decimal numbers, and multi-sentence paragraphs correctly.

In [9]:
from src.preprocessing import segment_sentences

sentences = segment_sentences(cleaned, pipeline)

print(f"Total sentences detected: {len(sentences)}\n")
print("── First 8 sentences ───────────────────────────────────────────────")
for i, sent in enumerate(sentences[:8], 1):
    print(f"  [{i}] {sent}")

Total sentences detected: 552

── First 8 sentences ───────────────────────────────────────────────
  [1] in order to better understand the nature of star formation occurring in nearby starburst galaxies and to provide a foundation for studies of higher redshift systems , we have been analyzing the properties of the 20 starburst galaxies with the highest 5 ghz luminosities from condon , frayer , & broderick ( 1991 ) .
  [2] these galaxies form a sample of relatively nearby , northern hemisphere starbursts which is independent of the dust content and the morphology of the host galaxy .
  [3] an example of our data and analysis technique is given in smith et al .
  [4] ( 1995 ) , hereafter paper 1 .
  [5] the complete sample and dataset are described in smith et al .
  [6] ( 1996 ) , hereafter paper 2 , and are analyzed in smith , herter , & haynes ( 1998 ) , hereafter paper 3 .
  [7] since our near infrared imaging data indicate the presence of strong central peaks of star formation , a

## 4. Stopword Removal

Removes common English stopwords (the, is, at, which, etc.) using NLTK.
The output is used downstream for keyword extraction — not for model input,
since transformer models handle stopwords internally.

In [10]:
from src.preprocessing import remove_stopwords

no_sw = remove_stopwords(cleaned, pipeline)

print("── BEFORE (first 300 chars) ─────────────────────────────────────────")
print(cleaned[:300])
print("\n── AFTER stopword removal (first 300 chars) ─────────────────────────")
print(no_sw[:300])
print(f"\nToken count before: {len(cleaned.split()):,}")
print(f"Token count after : {len(no_sw.split()):,}")
print(f"Stopwords removed : {len(cleaned.split()) - len(no_sw.split()):,}")

── BEFORE (first 300 chars) ─────────────────────────────────────────
in order to better understand the nature of star formation occurring in nearby starburst galaxies and to provide a foundation for studies of higher redshift systems , we have been analyzing the properties of the 20 starburst galaxies with the highest 5 ghz luminosities from condon , frayer , & brode

── AFTER stopword removal (first 300 chars) ─────────────────────────
order better understand nature star formation occurring nearby starburst galaxies provide foundation studies higher redshift systems analyzing properties 20 starburst galaxies highest ghz luminosities condon frayer broderick 1991 galaxies form sample relatively nearby northern hemisphere starbursts 

Token count before: 15,257
Token count after : 8,094
Stopwords removed : 7,163


## 5. Keyword Extraction

We use two complementary methods:

- **KeyBERT** — embedding-based. Finds phrases semantically similar to
  the document. Captures domain-specific terminology well.
- **YAKE** — statistical, unsupervised. Uses word position, frequency,
  and co-occurrence. Fast and requires no GPU.

We combine both into a deduplicated list for the best coverage.
"""

In [11]:
from src.preprocessing import (
    extract_keywords_keybert,
    extract_keywords_yake,
    extract_keywords_combined,
)

kb_kws   = extract_keywords_keybert(cleaned, pipeline, top_n=10)
yake_kws = extract_keywords_yake(cleaned, pipeline, top_n=10)
combined = extract_keywords_combined(cleaned, pipeline, top_n=10)

print("── KeyBERT keywords (keyword, relevance score) ──────────────────────")
for kw, score in kb_kws:
    print(f"  {kw:<35}  {score:.4f}")

print("\n── YAKE keywords (keyword, score — lower = more relevant) ───────────")
for kw, score in yake_kws:
    print(f"  {kw:<35}  {score:.6f}")

print("\n── Combined (deduplicated) ───────────────────────────────────────────")
for kw in combined:
    print(f"  {kw}")

── KeyBERT keywords (keyword, relevance score) ──────────────────────
  starburst galaxies                   0.7161
  galaxies infrared                    0.6708
  luminous galaxies                    0.6473
  starburst galaxy                     0.6353
  galaxies ngc                         0.6227
  observed starburst                   0.6159
  luminosity galaxies                  0.6088
  luminous starbursts                  0.6063
  starburst ngc                        0.6031
  observations starburst               0.5992

── YAKE keywords (keyword, score — lower = more relevant) ───────────
  radio emission                       0.000008
  star formation                       0.000009
  infrared emission                    0.000014
  thermal emission                     0.000026
  thermal radio                        0.000029
  infrared bright                      0.000038
  infrared colors                      0.000039
  bright knots                         0.000043
  radio bright 

## 6. Named Entity Recognition

spaCy's NER identifies real-world entities: organisations, people,
locations, dates, products. For medical papers this surfaces drug names,
diseases, and clinical terms (labelled as ORG or PRODUCT by en_core_web_sm).
A domain-specific model like en_core_sci_sm would do better on full medical
text — noted as future work.

In [12]:
from src.preprocessing import extract_entities

entities = extract_entities(cleaned, pipeline)

print("── Named Entities ───────────────────────────────────────────────────")
if entities:
    for label, ents in sorted(entities.items()):
        print(f"\n  [{label}]")
        for e in ents[:8]:   # show up to 8 per type
            print(f"    • {e}")
else:
    print("  No entities found in this sample.")

── Named Entities ───────────────────────────────────────────────────

  [CARDINAL]
    • 20
    • 5
    • 1
    • 2
    • 3
    • only two
    • 592
    • 0.9

  [DATE]
    • 1991
    • 1995
    • 1996
    • 1998
    • 7771
    • 12815
    • 1992
    • 7469

  [EVENT]
    • the 20.4 mjy

  [FAC]
    • @xmath189

  [GPE]
    • condon
    • @xmath72
    • relcontrib
    • @xmath142%
    • @xmath118%
    • @xmath157
    • mjy
    • @xmath178

  [LAW]
    • @xmath149
    • @xmath183

  [MONEY]
    • 483
    • 8a

  [NORP]
    • r.
    • @xmath105
    • @xmath120
    • @xmath148
    • @xmath200
    • @xmath208
    • @xmath211
    • @xmath213

  [ORDINAL]
    • first
    • second
    • third

  [ORG]
    • frayer , & broderick
    • smith et
    • neff &
    • al
    • devereux
    • kenney , & young
    • wilson , & baldwin
    • kitt peak

  [PERCENT]
    • 50%
    • 8%
    • 5%
    • 6%
    • 10%
    • 46%
    • 62%
    • 2%

  [PERSON]
    • al
    • maoz
    • @xmath9 km
    • s@xmath6

## 7. Full Pipeline — Single Entry Point

`preprocess()` runs all six steps in order and returns a single dict.
This is what every other notebook calls.

In [13]:
from src.preprocessing import preprocess

result = preprocess(sample["article"], pipeline)

print("── Full pipeline output keys ─────────────────────────────────────────")
for key, val in result.items():
    if isinstance(val, str):
        print(f"  {key:<15}: {len(val):>6,} chars   preview: {val[:80].strip()!r}")
    elif isinstance(val, list):
        print(f"  {key:<15}: {len(val):>6,} items")
    elif isinstance(val, dict):
        if key == "entities":
            # entities dict has list values, not strings
            found = [f"{label}({len(ents)})" for label, ents in val.items()]
            print(f"  {key:<15}: {len(val):>6,} types    found: {found}")
        else:
            # sections dict has string values
            found = [k for k, v in val.items() if isinstance(v, str) and v.strip()]
            print(f"  {key:<15}: {len(val):>6,} sections  found: {found}")

── Full pipeline output keys ─────────────────────────────────────────
  cleaned_text   : 81,657 chars   preview: 'in order to better understand the nature of star formation occurring in nearby s'
  sections       :      6 sections  found: ['introduction']
  sentences      :    552 items
  no_stopwords   : 58,737 chars   preview: 'order better understand nature star formation occurring nearby starburst galaxie'
  keywords       :     10 items
  entities       :     15 types    found: ['CARDINAL(63)', 'GPE(13)', 'ORG(100)', 'DATE(40)', 'PERSON(66)', 'QUANTITY(16)', 'MONEY(2)', 'NORP(11)', 'PRODUCT(5)', 'PERCENT(26)', 'EVENT(1)', 'ORDINAL(3)', 'WORK_OF_ART(1)', 'LAW(2)', 'FAC(1)']


In [ ]:
import json

print(f"Running full pipeline on {len(train_rows)} train rows ...")
print("(This may take 1–2 minutes in dev mode)\n")

preprocessed = []
for i, row in enumerate(train_rows):
    result = preprocess(row["article"], pipeline)
    preprocessed.append({
        "domain"       : row["domain"],
        "title"        : row.get("title", ""),
        "summary"      : row["summary"],
        "keywords"     : row.get("keywords", ""),
        **result,       # cleaned_text, sections, sentences, no_stopwords, keywords, entities
    })
    if (i + 1) % 50 == 0:
        print(f"  Processed {i+1}/{len(train_rows)}")

# Save
out_dir = f"../data/preprocessed_{DATASET_MODE}"
os.makedirs(out_dir, exist_ok=True)
out_path = f"{out_dir}/train_preprocessed.json"

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(preprocessed, f, ensure_ascii=False, indent=2)

print(f"\nSaved {len(preprocessed)} preprocessed rows → {out_path}")

Running full pipeline on 322961 train rows ...
(This may take 1–2 minutes in dev mode)

  Processed 50/322961
  Processed 100/322961
  Processed 150/322961
  Processed 200/322961
  Processed 250/322961
  Processed 300/322961
  Processed 350/322961
  Processed 400/322961
  Processed 450/322961
  Processed 500/322961
  Processed 550/322961
  Processed 600/322961
  Processed 650/322961
  Processed 700/322961
  Processed 750/322961
  Processed 800/322961
  Processed 850/322961
  Processed 900/322961
  Processed 950/322961
  Processed 1000/322961
  Processed 1050/322961
  Processed 1100/322961
  Processed 1150/322961
  Processed 1200/322961
  Processed 1250/322961
  Processed 1300/322961
  Processed 1350/322961
  Processed 1400/322961
  Processed 1450/322961
  Processed 1500/322961
  Processed 1550/322961
  Processed 1600/322961
  Processed 1650/322961
  Processed 1700/322961
  Processed 1750/322961
  Processed 1800/322961
  Processed 1850/322961
  Processed 1900/322961
  Processed 1950/322

## ✅ Phase 2 Complete — Gate Check

- [ ] `src/preprocessing.py` created with all functions and docstrings
- [ ] Text cleaning removes LaTeX, URLs, citations correctly
- [ ] Section extraction finds at least some sections (more visible on full data)
- [ ] Sentence segmentation produces sensible sentence boundaries
- [ ] Stopword removal reduces token count meaningfully
- [ ] KeyBERT + YAKE both return domain-relevant keywords
- [ ] NER finds entities in the sample text
- [ ] Full pipeline runs on all train rows without errors
- [ ] Preprocessed data saved to `data/preprocessed_dev/`

Proceed to `03_domain_classifier.ipynb`.